In [4]:
# -*- coding: utf-8 -*-
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from PIL import Image

# Preprocessing & Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier

# Model selection & metrics
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, matthews_corrcoef, roc_auc_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

# ---------------------- Configuration ----------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TRAIN_DATASET_PATH = "train_dataset.csv"
TEST_DATASET_PATH = "test_dataset.csv"
TARGET_COLUMN = "Activity_Label"
RFE_FEATURES_FILE = "rfe_selected_features.csv"

BASE_DIR = "mic_activity_prediction_study_v2"
MODEL_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
CSV_DIR = os.path.join(RESULTS_DIR, "csv")

for dir_path in [MODEL_DIR, RESULTS_DIR, FIGURES_DIR, CSV_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# ---------------------- Load Data ----------------------
train_df = pd.read_csv(TRAIN_DATASET_PATH)
test_df = pd.read_csv(TEST_DATASET_PATH)

y_train = train_df[TARGET_COLUMN]
y_test = test_df[TARGET_COLUMN]

X_train_all = train_df.drop(columns=[TARGET_COLUMN])
X_test_all = test_df.drop(columns=[TARGET_COLUMN])

# ---------------------- Load RFE Features ----------------------
if not os.path.exists(RFE_FEATURES_FILE):
    raise FileNotFoundError(f"{RFE_FEATURES_FILE} not found. Run RFE first.")

rfe_features = pd.read_csv(RFE_FEATURES_FILE)["RFE_Selected_Features"].tolist()
missing_features = [f for f in rfe_features if f not in X_train_all.columns]
if missing_features:
    raise ValueError(f"Features missing in dataset: {missing_features}")

X_train_filtered = X_train_all[rfe_features].copy()
X_test_filtered = X_test_all[rfe_features].copy()

# Clean infinities/NaNs
for df_ in [X_train_filtered, X_test_filtered]:
    df_.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_.fillna(df_.mean(), inplace=True)
    df_ = df_.astype(float)

# ---------------------- Visualizations ----------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_filtered)

# PCA
X_pca = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y_train, palette={0:"blue",1:"red"})
plt.title("PCA of RFE-selected features")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR,"PCA_plot.png")); plt.close()

# t-SNE
X_tsne = TSNE(n_components=2, random_state=RANDOM_STATE, learning_rate='auto', init='random').fit_transform(X_scaled)
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_tsne[:,0], y=X_tsne[:,1], hue=y_train, palette={0:"blue",1:"red"})
plt.title("t-SNE of RFE-selected features")
plt.xlabel("tSNE1"); plt.ylabel("tSNE2")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR,"tSNE_plot.png")); plt.close()

# UMAP
X_umap = umap.UMAP(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_umap[:,0], y=X_umap[:,1], hue=y_train, palette={0:"blue",1:"red"})
plt.title("UMAP of RFE-selected features")
plt.xlabel("UMAP1"); plt.ylabel("UMAP2")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR,"UMAP_plot.png")); plt.close()

# ---------------------- Pipelines ----------------------
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
median_imputer = SimpleImputer(strategy="median")

pipelines = {}
param_grids = {}

# kNN
pipelines['kNN'] = Pipeline([('imputer', median_imputer), ('scaler', scaler), ('model', KNeighborsClassifier())])
param_grids['kNN'] = [{'model__n_neighbors': list(range(1,28,2))}]

# MLP
pipelines['MLP'] = Pipeline([('imputer', median_imputer), ('scaler', scaler),
                             ('model', MLPClassifier(random_state=RANDOM_STATE, early_stopping=True, validation_fraction=0.1))])
param_grids['MLP'] = [{'model__hidden_layer_sizes': [(n,) for n in range(20,101,20)]+[(50,25),(100,50)],
                       'model__activation':['relu','tanh'],'model__alpha':[0.0001,0.001,0.01],'model__learning_rate':['constant','adaptive'],'model__max_iter':[1500]}]

# Naive Bayes
pipelines['Naive Bayes'] = Pipeline([('imputer', median_imputer), ('model', GaussianNB())])
param_grids['Naive Bayes'] = [{'model__var_smoothing': np.logspace(-9,-2,50)}]

# Decision Tree
pipelines['Decision Tree'] = Pipeline([('imputer', median_imputer), ('model', DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'))])
param_grids['Decision Tree'] = [{'model__criterion': ['gini','entropy'],
                                 'model__min_samples_split':[2,5,10],'model__min_samples_leaf':[1,5,10],'model__max_depth':[None,5,10,20]}]

# Random Forest
pipelines['Random Forest'] = Pipeline([('imputer', median_imputer), ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'))])
param_grids['Random Forest'] = [{'model__n_estimators':[100,300],'model__max_depth':[10,20],'model__min_samples_split':[5,10],'model__min_samples_leaf':[3,5],'model__criterion':['gini','entropy']}]

# CatBoost
pipelines['CatBoost'] = Pipeline([('imputer', median_imputer),
                                  ('model', CatBoostClassifier(random_state=RANDOM_STATE, verbose=0,class_weights='Balanced'))])
param_grids['CatBoost'] = [{'model__iterations':[300,500],'model__depth':[4,6,8],'model__learning_rate':[0.05,0.1]}]

models_to_process = list(pipelines.keys())

# ---------------------- Training & Evaluation ----------------------
results = {}
best_params = {}
classification_reports_dict = {}
SCORING_METRIC = 'matthews_corrcoef'

for name in models_to_process:
    print(f"\nProcessing {name}...")
    pipeline = pipelines[name]
    param_grid = param_grids[name] if name in param_grids else [{}]

    try:
        grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=cv_strategy,
                                   scoring=SCORING_METRIC, n_jobs=4, refit=True)
        grid_search.fit(X_train_filtered, y_train)
        best_pipeline = grid_search.best_estimator_
        best_params[name] = grid_search.best_params_

        joblib.dump(best_pipeline, os.path.join(MODEL_DIR, f"{name}_best_pipeline.pkl"))

        y_pred = best_pipeline.predict(X_test_filtered)
        y_pred_prob = best_pipeline.predict_proba(X_test_filtered)[:,1] if hasattr(best_pipeline.steps[-1][1], "predict_proba") else None

        test_acc = accuracy_score(y_test, y_pred)
        test_mcc = matthews_corrcoef(y_test, y_pred)
        test_auc = roc_auc_score(y_test, y_pred_prob) if y_pred_prob is not None else "N/A"

        results[name] = {"Test Accuracy": round(test_acc,4),"Test MCC": round(test_mcc,4),"Test AUC": test_auc}

        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(5,4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Pred Inactive","Pred Active"],
                    yticklabels=["Actual Inactive","Actual Active"])
        plt.title(f"Confusion Matrix - {name}")
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR,f"{name}_confusion_matrix.png"))
        plt.close()

    except Exception as e:
        print(f"Error training {name}: {e}")
        results[name] = {"Test Accuracy":"Error","Test MCC":"Error","Test AUC":"Error"}
        best_params[name] = "Error"



/opt/conda/envs/openmm-env/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(



Processing kNN...

Processing MLP...

Processing Naive Bayes...

Processing Decision Tree...

Processing Random Forest...

Processing CatBoost...
Error training CatBoost: 
All the 60 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
60 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/conda/envs/openmm-env/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/openmm-env/lib/python3.13/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/conda/envs/openmm-env/lib/python3.13/site-packages/sklearn/pipeline.py", line 663, in f

In [13]:
# -*- coding: utf-8 -*-
"""
QSAR Model Training and Evaluation Script.

This script loads pre-selected features, visualizes the data distribution,
trains multiple classification models using GridSearchCV, evaluates their
performance on a test set, and saves the results, models, and figures.
"""
import os
import json
import warnings
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Preprocessing & Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Dimensionality Reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

# Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# Model selection & metrics
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, matthews_corrcoef, roc_auc_score, 
                             confusion_matrix, f1_score, balanced_accuracy_score)

# ---------------------- Configuration ----------------------
# Core Settings
RANDOM_STATE = 42
CV_SPLITS = 5
N_JOBS = -1  # Use all available CPU cores
SCORING_METRIC = 'matthews_corrcoef'
TARGET_COLUMN = "Activity_Label"

# File Paths
TRAIN_DATASET_PATH = "train_dataset.csv"
TEST_DATASET_PATH = "test_dataset.csv"
RFE_FEATURES_FILE = "rfe_selected_features.csv"

# Output Directories
BASE_DIR = "mic_activity_prediction_study_v3"  # New version for improved script
MODEL_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
CSV_DIR = os.path.join(RESULTS_DIR, "csv")

# ---------------------- Helper Functions ----------------------

def setup_directories():
    """Create all necessary output directories."""
    print("Setting up output directories...")
    for dir_path in [MODEL_DIR, RESULTS_DIR, FIGURES_DIR, CSV_DIR]:
        os.makedirs(dir_path, exist_ok=True)

def load_and_prepare_data():
    """Load train/test data and filter using RFE features."""
    print("Loading and preparing data...")
    # Load datasets
    train_df = pd.read_csv(TRAIN_DATASET_PATH)
    test_df = pd.read_csv(TEST_DATASET_PATH)

    y_train = train_df[TARGET_COLUMN]
    y_test = test_df[TARGET_COLUMN]

    # Load and validate RFE-selected features
    if not os.path.exists(RFE_FEATURES_FILE):
        raise FileNotFoundError(f"{RFE_FEATURES_FILE} not found. Please run RFE first.")
    
    rfe_features = pd.read_csv(RFE_FEATURES_FILE)["RFE_Selected_Features"].tolist()
    
    X_train = train_df[rfe_features]
    X_test = test_df[rfe_features]
    
    # Handle potential infinity values from descriptor calculation
    X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    print(f"Data loaded successfully with {X_train.shape[1]} RFE-selected features.")
    return X_train, y_train, X_test, y_test

def generate_embedding_plot(X: pd.DataFrame, y: pd.Series, method: str):
    """Generate and save a 2D embedding plot (PCA, t-SNE, or UMAP)."""
    print(f"Generating {method.upper()} plot...")
    
    # Scale data before dimensionality reduction
    X_scaled = StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(X))
    
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=RANDOM_STATE)
        components = reducer.fit_transform(X_scaled)
        x_label, y_label = "PC1", "PC2"
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=RANDOM_STATE, init='pca', learning_rate='auto')
        components = reducer.fit_transform(X_scaled)
        x_label, y_label = "tSNE1", "tSNE2"
    elif method == 'umap':
        reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
        components = reducer.fit_transform(X_scaled)
        x_label, y_label = "UMAP1", "UMAP2"
    else:
        raise ValueError("Method must be 'pca', 'tsne', or 'umap'.")

    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=y, palette={0: "blue", 1: "red"})
    plt.title(f"{method.upper()} of RFE-selected Features")
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{method.upper()}_plot.png"))
    plt.close()

def define_pipelines_and_grids() -> (Dict[str, Pipeline], Dict[str, List[Dict[str, Any]]]):
    """Defines the model pipelines and hyperparameter grids for GridSearchCV."""
    median_imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    
    pipelines = {
        'kNN': Pipeline([('imputer', median_imputer), ('scaler', scaler), ('model', KNeighborsClassifier())]),
        
        'MLP': Pipeline([('imputer', median_imputer), ('scaler', scaler), ('model', MLPClassifier(random_state=RANDOM_STATE, early_stopping=True, validation_fraction=0.1))]),
        
        'Naive Bayes': Pipeline([('imputer', median_imputer), ('model', GaussianNB())]),
        
        'Decision Tree': Pipeline([('imputer', median_imputer), ('model', DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'))]),
        
        'Random Forest': Pipeline([('imputer', median_imputer), ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=N_JOBS))]),
        
        'CatBoost': Pipeline([('imputer', median_imputer), ('model', CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, auto_class_weights='Balanced'))]),
                
        'SVM': Pipeline([('imputer', median_imputer), ('scaler', scaler), ('model', SVC(random_state=RANDOM_STATE, probability=True, class_weight='balanced'))]),
        
        'Logistic Regression': Pipeline([('imputer', median_imputer), ('scaler', scaler), ('model', LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', max_iter=1000, n_jobs=N_JOBS))])
    }

    param_grids = {
        'kNN': [{'model__n_neighbors': list(range(1, 28, 2))}],
        
        'MLP': [{'model__hidden_layer_sizes': [(n,) for n in range(20, 101, 20)] + [(50, 25), (100, 50)],
                 'model__activation': ['relu', 'tanh'], 'model__alpha': [0.0001, 0.001, 0.01],
                 'model__learning_rate': ['constant', 'adaptive'], 'model__max_iter': [1500]}],
                 
        'Naive Bayes': [{'model__var_smoothing': np.logspace(-9, -2, 50)}],
        
        'Decision Tree': [{'model__criterion': ['gini', 'entropy'], 'model__min_samples_split': [2, 5, 10],
                           'model__min_samples_leaf': [1, 5, 10], 'model__max_depth': [None, 5, 10, 20]}],
                           
        'Random Forest': [{'model__n_estimators': [100, 300, 500], 'model__max_depth': [10, 20, None],
                           'model__min_samples_split': [5, 10], 'model__min_samples_leaf': [3, 5]}],
                           
        'CatBoost': [{'model__iterations': [300, 500], 'model__depth': [4, 6, 8], 'model__learning_rate': [0.05, 0.1]}],
                      
        'SVM': [{'model__C': [0.1, 1, 10], 'model__gamma': ['scale', 'auto', 0.1, 1], 'model__kernel': ['rbf']}],
        
        'Logistic Regression': [{'model__penalty': ['l1', 'l2'], 'model__C': np.logspace(-3, 3, 7), 'model__solver': ['liblinear']}]
    }
    return pipelines, param_grids

def train_and_evaluate_models(X_train, y_train, X_test, y_test, pipelines, param_grids):
    """Train all models, evaluate on test set, and save artifacts."""
    results_list = []
    best_params = {}
    cv_strategy = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for name, pipeline in pipelines.items():
        print(f"\n--- Processing {name} ---")
        param_grid = param_grids.get(name, [{}])

        try:
            # Grid Search with Cross-Validation
            grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=cv_strategy,
                                       scoring=SCORING_METRIC, n_jobs=N_JOBS, refit=True)
            grid_search.fit(X_train, y_train)
            best_pipeline = grid_search.best_estimator_
            best_params[name] = grid_search.best_params_

            # Save the best model
            joblib.dump(best_pipeline, os.path.join(MODEL_DIR, f"{name}_best_model.pkl"))

            # Evaluate on the test set
            y_pred = best_pipeline.predict(X_test)
            y_prob = best_pipeline.predict_proba(X_test)[:, 1] if hasattr(best_pipeline, "predict_proba") else [0] * len(y_test)
            
            # Store results, including the new metrics
            results_list.append({
                'Model': name,
                'Test Matthews Corrcoef': matthews_corrcoef(y_test, y_pred),
                'Test Balanced Accuracy': balanced_accuracy_score(y_test, y_pred), # ADDED
                'Test F1 Score': f1_score(y_test, y_pred),                         # ADDED
                'Test AUC': roc_auc_score(y_test, y_prob),
                'Test Accuracy': accuracy_score(y_test, y_pred)
            })

            # Generate and save confusion matrix (no changes needed here)
            cm = confusion_matrix(y_test, y_pred)
            plt.figure(figsize=(5, 4))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                        xticklabels=["Pred Inactive", "Pred Active"],
                        yticklabels=["Actual Inactive", "Actual Active"])
            plt.title(f"Confusion Matrix - {name}")
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, f"{name}_confusion_matrix.png"))
            plt.close()

        except Exception as e:
            print(f"Error training {name}: {e}")
            results_list.append({
                'Model': name, 
                'Test Matthews Corrcoef': 'Error', 
                'Test Balanced Accuracy': 'Error', # ADDED
                'Test F1 Score': 'Error',         # ADDED
                'Test AUC': 'Error', 
                'Test Accuracy': 'Error'
            })
            best_params[name] = "Error"
    
    return pd.DataFrame(results_list), best_params

# ---------------------- Main Execution ----------------------

def main():
    """Main function to run the entire ML pipeline."""
    np.random.seed(RANDOM_STATE)
    
    setup_directories()
    X_train, y_train, X_test, y_test = load_and_prepare_data()
    
    # Run and save visualizations
    for method in ['pca', 'tsne', 'umap']:
        generate_embedding_plot(X_train, y_train, method)
        
    pipelines, param_grids = define_pipelines_and_grids()
    results_df, best_params = train_and_evaluate_models(X_train, y_train, X_test, y_test, pipelines, param_grids)
    
    # --- Final Reporting ---
    print("\n\n--- Final Model Performance Summary ---")
    results_df = results_df.sort_values(by=f'Test {SCORING_METRIC.replace("_", " ").title()}', ascending=False)
    print(results_df.to_string(index=False))

    # Save summary results and best parameters
    results_df.to_csv(os.path.join(CSV_DIR, "model_performance_summary.csv"), index=False)
    with open(os.path.join(RESULTS_DIR, "best_hyperparameters.json"), 'w') as f:
        json.dump(best_params, f, indent=4)
        
    print(f"\n✅ Pipeline complete. All results saved in '{BASE_DIR}' directory.")


if __name__ == "__main__":
    main()


Setting up output directories...
Loading and preparing data...
Data loaded successfully with 20 RFE-selected features.
Generating PCA plot...
Generating TSNE plot...
Generating UMAP plot...

--- Processing kNN ---

--- Processing MLP ---

--- Processing Naive Bayes ---

--- Processing Decision Tree ---

--- Processing Random Forest ---

--- Processing CatBoost ---

--- Processing SVM ---

--- Processing Logistic Regression ---


--- Final Model Performance Summary ---
              Model  Test Matthews Corrcoef  Test Balanced Accuracy  Test F1 Score  Test AUC  Test Accuracy
                SVM                0.612977                0.851190       0.647887  0.904881       0.919355
                kNN                0.582016                0.782143       0.620690  0.894405       0.929032
      Random Forest                0.572753                0.806548       0.615385  0.907976       0.919355
        Naive Bayes                0.533840                0.799405       0.579710  0.859048   

In [1]:
# -*- coding: utf-8 -*-
"""
QSAR Model Training and Evaluation Script.

This script loads pre-selected features, visualizes the data distribution,
trains multiple classification models using GridSearchCV, evaluates their
performance on a test set, and saves the results, models, and figures.
"""
import os
import json
import warnings
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Preprocessing & Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Dimensionality Reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

# Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# Model selection & metrics
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, matthews_corrcoef, roc_auc_score,
                             confusion_matrix, f1_score, balanced_accuracy_score)

# ---------------------- Configuration ----------------------
# Core Settings
RANDOM_STATE = 42
CV_SPLITS = 5
N_JOBS = -1  # Use all available CPU cores
SCORING_METRIC = 'matthews_corrcoef'
TARGET_COLUMN = "Activity_Label"

# File Paths
TRAIN_DATASET_PATH = "train_dataset.csv"
TEST_DATASET_PATH = "test_dataset.csv"
RFE_FEATURES_FILE = "rfe_selected_features.csv"

# Output Directories
BASE_DIR = "mic_activity_prediction_study_v4"  # New version for improved script
MODEL_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
CSV_DIR = os.path.join(RESULTS_DIR, "csv")

# ---------------------- Helper Functions ----------------------

def setup_directories():
    """Create all necessary output directories."""
    print("Setting up output directories...")
    for dir_path in [MODEL_DIR, RESULTS_DIR, FIGURES_DIR, CSV_DIR]:
        os.makedirs(dir_path, exist_ok=True)

def load_and_prepare_data():
    """Load train/test data and filter using RFE features."""
    print("Loading and preparing data...")
    # Load datasets
    train_df = pd.read_csv(TRAIN_DATASET_PATH)
    test_df = pd.read_csv(TEST_DATASET_PATH)

    y_train = train_df[TARGET_COLUMN]
    y_test = test_df[TARGET_COLUMN]

    # Load and validate RFE-selected features
    if not os.path.exists(RFE_FEATURES_FILE):
        raise FileNotFoundError(f"{RFE_FEATURES_FILE} not found. Please run RFE first.")
    
    rfe_features = pd.read_csv(RFE_FEATURES_FILE)["RFE_Selected_Features"].tolist()
    
    X_train = train_df[rfe_features]
    X_test = test_df[rfe_features]
    
    # Handle potential infinity values from descriptor calculation
    X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    print(f"Data loaded successfully with {X_train.shape[1]} RFE-selected features.")
    return X_train, y_train, X_test, y_test

def generate_embedding_plot(X: pd.DataFrame, y: pd.Series, method: str):
    """Generate and save a 2D embedding plot (PCA, t-SNE, or UMAP)."""
    print(f"Generating {method.upper()} plot...")
    
    # Scale data before dimensionality reduction
    X_scaled = StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(X))
    
    if method == 'pca':
        reducer = PCA(n_components=2, random_state=RANDOM_STATE)
        components = reducer.fit_transform(X_scaled)
        x_label, y_label = "PC1", "PC2"
    elif method == 'tsne':
        reducer = TSNE(n_components=2, random_state=RANDOM_STATE, init='pca', learning_rate='auto')
        components = reducer.fit_transform(X_scaled)
        x_label, y_label = "tSNE1", "tSNE2"
    elif method == 'umap':
        reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE, n_neighbors=15, min_dist=0.1)
        components = reducer.fit_transform(X_scaled)
        x_label, y_label = "UMAP1", "UMAP2"
    else:
        raise ValueError("Method must be 'pca', 'tsne', or 'umap'.")

    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=y, palette={0: "blue", 1: "red"})
    plt.title(f"{method.upper()} of RFE-selected Features")
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, f"{method.upper()}_plot.png"))
    plt.close()

def define_pipelines_and_grids() -> (Dict[str, Pipeline], Dict[str, List[Dict[str, Any]]]):
    """Defines the model pipelines and hyperparameter grids for GridSearchCV."""
    median_imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    
    pipelines = {
        'kNN': Pipeline([('imputer', median_imputer), ('scaler', scaler), ('model', KNeighborsClassifier())]),
        
        'MLP': Pipeline([('imputer', median_imputer), ('scaler', scaler), ('model', MLPClassifier(random_state=RANDOM_STATE, early_stopping=True, validation_fraction=0.1))]),
        
        'Naive Bayes': Pipeline([('imputer', median_imputer), ('model', GaussianNB())]),
        
        'Decision Tree': Pipeline([('imputer', median_imputer), ('model', DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'))]),
        
        'Random Forest': Pipeline([('imputer', median_imputer), ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=N_JOBS))]),
        
        'CatBoost': Pipeline([('imputer', median_imputer), ('model', CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, auto_class_weights='Balanced'))]),
              
        'SVM': Pipeline([('imputer', median_imputer), ('scaler', scaler), ('model', SVC(random_state=RANDOM_STATE, probability=True, class_weight='balanced'))]),
        
        'Logistic Regression': Pipeline([('imputer', median_imputer), ('scaler', scaler), ('model', LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', max_iter=1000, n_jobs=N_JOBS))])
    }

    param_grids = {
        'kNN': [{'model__n_neighbors': list(range(1, 28, 2))}],
        
        'MLP': [{'model__hidden_layer_sizes': [(n,) for n in range(20, 101, 20)] + [(50, 25), (100, 50)],
                 'model__activation': ['relu', 'tanh'], 'model__alpha': [0.0001, 0.001, 0.01],
                 'model__learning_rate': ['constant', 'adaptive'], 'model__max_iter': [1500]}],
                 
        'Naive Bayes': [{'model__var_smoothing': np.logspace(-9, -2, 50)}],
        
        'Decision Tree': [{'model__criterion': ['gini', 'entropy'], 'model__min_samples_split': [2, 5, 10],
                           'model__min_samples_leaf': [1, 5, 10], 'model__max_depth': [None, 5, 10, 20]}],
                           
        'Random Forest': [{'model__n_estimators': [100, 300, 500], 'model__max_depth': [10, 20, None],
                           'model__min_samples_split': [5, 10], 'model__min_samples_leaf': [3, 5]}],
                           
        'CatBoost': [{'model__iterations': [300, 500], 'model__depth': [4, 6, 8], 'model__learning_rate': [0.05, 0.1]}],
                       
        'SVM': [{'model__C': [0.1, 1, 10], 'model__gamma': ['scale', 'auto', 0.1, 1], 'model__kernel': ['rbf']}],
        
        'Logistic Regression': [{'model__penalty': ['l1', 'l2'], 'model__C': np.logspace(-3, 3, 7), 'model__solver': ['liblinear']}]
    }
    return pipelines, param_grids

def train_and_evaluate_models(X_train, y_train, X_test, y_test, pipelines, param_grids):
    """Train all models, evaluate on test set, and save artifacts."""
    results_list = []
    best_params = {}
    cv_strategy = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for name, pipeline in pipelines.items():
        print(f"\n--- Processing {name} ---")
        param_grid = param_grids.get(name, [{}])

        try:
            # Grid Search with Cross-Validation
            grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=cv_strategy,
                                       scoring=SCORING_METRIC, n_jobs=N_JOBS, refit=True)
            grid_search.fit(X_train, y_train)
            best_pipeline = grid_search.best_estimator_
            best_params[name] = grid_search.best_params_

            # Save the best model
            joblib.dump(best_pipeline, os.path.join(MODEL_DIR, f"{name}_best_model.pkl"))

            # Evaluate on the test set
            y_pred = best_pipeline.predict(X_test)
            y_prob = best_pipeline.predict_proba(X_test)[:, 1] if hasattr(best_pipeline, "predict_proba") else [0] * len(y_test)
            
            # Store results
            results_list.append({
                'Model': name,
                'Test Matthews Corrcoef': matthews_corrcoef(y_test, y_pred),
                'Test Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
                'Test F1 Score': f1_score(y_test, y_pred),
                'Test AUC': roc_auc_score(y_test, y_prob),
                'Test Accuracy': accuracy_score(y_test, y_pred)
            })

            # Generate and save confusion matrix
            cm = confusion_matrix(y_test, y_pred)
            plt.figure(figsize=(5, 4))
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                        xticklabels=["Pred Inactive", "Pred Active"],
                        yticklabels=["Actual Inactive", "Actual Active"])
            plt.title(f"Confusion Matrix - {name}")
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, f"{name}_confusion_matrix.png"))
            plt.close()

        except Exception as e:
            print(f"Error training {name}: {e}")
            results_list.append({
                'Model': name, 
                'Test Matthews Corrcoef': 'Error', 
                'Test Balanced Accuracy': 'Error',
                'Test F1 Score': 'Error',
                'Test AUC': 'Error', 
                'Test Accuracy': 'Error'
            })
            best_params[name] = "Error"
    
    return pd.DataFrame(results_list), best_params

def generate_performance_comparison_plot(results_df: pd.DataFrame):
    """Generate and save a bar plot comparing the performance of all models."""
    print("\nGenerating model performance comparison plot...")

    # Create a copy to avoid modifying the original DataFrame
    plot_df = results_df.copy()

    # Ensure metric columns are numeric, coercing errors to NaN
    for col in plot_df.columns:
        if col != 'Model':
            plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')
    plot_df.dropna(inplace=True) # Drop rows where an error occurred

    if plot_df.empty:
        print("No valid data to plot after cleaning. Skipping comparison plot.")
        return

    # Melt the DataFrame to long format for easier plotting with Seaborn
    df_melted = plot_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

    plt.figure(figsize=(15, 8))
    sns.barplot(data=df_melted, x='Model', y='Score', hue='Metric', palette='viridis')

    plt.title('Model Performance Comparison on Test Set', fontsize=16, fontweight='bold')
    plt.xlabel('Model', fontsize=12)
    plt.ylabel('Score', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1.05)
    plt.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout(rect=[0, 0, 0.85, 1]) # Adjust layout to make space for legend

    # Save the figure
    plot_path = os.path.join(FIGURES_DIR, "model_performance_comparison.png")
    plt.savefig(plot_path)
    plt.close()
    print(f"Comparison plot saved to {plot_path}")

# ---------------------- Main Execution ----------------------

def main():
    """Main function to run the entire ML pipeline."""
    np.random.seed(RANDOM_STATE)
    
    setup_directories()
    X_train, y_train, X_test, y_test = load_and_prepare_data()
    
    # Run and save visualizations
    for method in ['pca', 'tsne', 'umap']:
        generate_embedding_plot(X_train, y_train, method)
        
    pipelines, param_grids = define_pipelines_and_grids()
    results_df, best_params = train_and_evaluate_models(X_train, y_train, X_test, y_test, pipelines, param_grids)
    
    # --- Generate Performance Comparison Plot ---
    generate_performance_comparison_plot(results_df)

    # --- Final Reporting ---
    print("\n\n--- Final Model Performance Summary ---")
    # Sort by the primary scoring metric
    sort_col = f'Test {SCORING_METRIC.replace("_", " ").title()}'
    if sort_col in results_df.columns:
         # Convert to numeric for sorting, coercing errors
        results_df[sort_col] = pd.to_numeric(results_df[sort_col], errors='coerce')
        results_df = results_df.sort_values(by=sort_col, ascending=False)

    print(results_df.to_string(index=False))

    # Save summary results and best parameters
    results_df.to_csv(os.path.join(CSV_DIR, "model_performance_summary.csv"), index=False)
    with open(os.path.join(RESULTS_DIR, "best_hyperparameters.json"), 'w') as f:
        json.dump(best_params, f, indent=4)
        
    print(f"\n✅ Pipeline complete. All results saved in '{BASE_DIR}' directory.")


if __name__ == "__main__":
    main()

Setting up output directories...
Loading and preparing data...
Data loaded successfully with 20 RFE-selected features.
Generating PCA plot...
Generating TSNE plot...
Generating UMAP plot...

--- Processing kNN ---

--- Processing MLP ---

--- Processing Naive Bayes ---

--- Processing Decision Tree ---

--- Processing Random Forest ---

--- Processing CatBoost ---

--- Processing SVM ---

--- Processing Logistic Regression ---

Generating model performance comparison plot...
Comparison plot saved to mic_activity_prediction_study_v4/results/figures/model_performance_comparison.png


--- Final Model Performance Summary ---
              Model  Test Matthews Corrcoef  Test Balanced Accuracy  Test F1 Score  Test AUC  Test Accuracy
                SVM                0.612977                0.851190       0.647887  0.904881       0.919355
                kNN                0.582016                0.782143       0.620690  0.894405       0.929032
      Random Forest                0.572753    